In [1]:
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.preprocessing import StandardScaler

# 1. Данные и Нормализация (в одну строку через pandas)
train = pd.read_csv('/kaggle/input/kazakhstan-respa-final-day-2-late-competition/train_sentences.csv')
test = pd.read_csv('/kaggle/input/kazakhstan-respa-final-day-2-late-competition/test_sentences.csv')
#train = train.sample(frac=0.5, random_state=42).reset_index(drop=True)

2026-02-10 05:57:15.517215: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770703035.730140      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770703035.796413      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770703036.342095      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770703036.342134      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770703036.342137      55 computation_placer.cc:177] computation placer alr

In [2]:
train['label'] = StandardScaler().fit_transform(
    pd.to_datetime(train['submitted_date']).apply(lambda x: x.toordinal()).values.reshape(-1, 1)
).flatten()

In [3]:
MODEL_NAME = 'distilroberta-base' 
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize(batch): 
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=64)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [4]:
train_ds = Dataset.from_pandas(train[['sentences', 'label']].rename(columns={'sentences': 'text'}))
train_ds = train_ds.map(tokenize, batched=True)

Map:   0%|          | 0/37177 [00:00<?, ? examples/s]

In [5]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1)

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
for param in model.base_model.parameters():
    param.requires_grad = False

In [7]:
training_args = TrainingArguments(
    output_dir='checkpoints',
    num_train_epochs=3,              # 3 эпохи пролетят быстро
    per_device_train_batch_size=128, # Большой батч, т.к. длина 64 и модель заморожена
    per_device_eval_batch_size=256,
    fp16=True,                       # Mixed Precision (обязательно для GPU T4/P100)
    dataloader_num_workers=2,        # Быстрая подгрузка данных
    report_to="none",
    save_strategy="no",              # Не тратим время на сохранение чекпоинтов
    logging_steps=100
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds
)

print("Начинаем турбо-обучение...")
trainer.train()

# --- Предсказание и Сабмит (как раньше) ---
def predict(texts):
    ds = Dataset.from_dict({'text': texts}).map(tokenize, batched=True)
    return trainer.predict(ds).predictions.flatten()

pred1 = predict(test['first_sentence'].tolist())
pred2 = predict(test['second_sentence'].tolist())

sub = pd.read_csv('/kaggle/input/kazakhstan-respa-final-day-2-late-competition/sample_submission.csv')
sub['label'] = (pred1 > pred2).astype(int)
sub.to_csv("submission_fast.csv", index=False)

Начинаем турбо-обучение...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
100,0.991000
200,0.958600
300,0.946400
400,0.936800


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Map:   0%|          | 0/24050 [00:00<?, ? examples/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Map:   0%|          | 0/24050 [00:00<?, ? examples/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [8]:
sub['label'] = (pred1 < pred2).astype(int)
sub.to_csv("submission_fast.csv", index=False)

In [1]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# --- 1. Данные и Препроцессинг ---
train = pd.read_csv('/kaggle/input/kazakhstan-respa-final-day-2-late-competition/train_sentences.csv')
test = pd.read_csv('/kaggle/input/kazakhstan-respa-final-day-2-late-competition/test_sentences.csv')

# Используем MinMaxScaler, он часто лучше для дат, чем StandardScaler, так как сохраняет диапазон
# Но можно оставить и StandardScaler, если работает.
# Важно: переводим дату в число.
scaler = StandardScaler()
train['timestamp'] = pd.to_datetime(train['submitted_date']).apply(lambda x: x.toordinal()).values.reshape(-1, 1)
train['label'] = scaler.fit_transform(train[['timestamp']]).flatten()

# --- 2. Настройки модели (УЛУЧШЕНИЕ) ---
# roberta-base лучше чем distilroberta. 
# Если тексты на русском/казахском, лучше взять 'xlm-roberta-base'
MODEL_NAME = 'roberta-base' 
MAX_LEN = 128 # Увеличили длину контекста

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=MAX_LEN)

# Готовим датасет. Важно: переименовываем колонку с текстом в 'text'
# Проверь, как называется колонка в твоем csv. Обычно 'sentence' или 'text'.
# В твоем коде было 'sentences', оставляю так.
train_ds = Dataset.from_pandas(train[['sentences', 'label']].rename(columns={'sentences': 'text'}))
train_ds = train_ds.map(tokenize, batched=True)

# Инициализация модели
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1)

# !!! ВАЖНО: УБРАЛИ ЗАМОРОЗКУ ВЕСОВ (param.requires_grad = False) !!!
# Теперь модель будет обучаться.

# --- 3. Аргументы обучения ---
training_args = TrainingArguments(
    output_dir='checkpoints_accurate',
    num_train_epochs=3,              # Можно увеличить до 4-5, если есть время
    learning_rate=2e-5,              # Маленький LR для точной настройки
    per_device_train_batch_size=16,  # Уменьшаем батч, т.к. модель разморожена и длиннее (память GPU)
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,   # Накапливаем градиенты, чтобы компенсировать малый батч
    fp16=True,                       # Ускорение
    weight_decay=0.01,               # Регуляризация от переобучения
    warmup_ratio=0.1,                # Прогрев
    logging_steps=50,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds
)

print("Начинаем полноценное обучение (Fine-tuning)...")
trainer.train()

# --- 4. Предсказание ---
def predict(texts):
    # Создаем датасет из списка текстов
    ds = Dataset.from_dict({'text': texts})
    ds = ds.map(tokenize, batched=True)
    # Получаем предсказания
    preds = trainer.predict(ds).predictions.flatten()
    return preds

print("Предсказание для test...")
# Предсказываем "дату" (число) для первого и второго предложения
pred1 = predict(test['first_sentence'].tolist())
pred2 = predict(test['second_sentence'].tolist())

# --- 5. Формирование сабмита ---
sub = pd.read_csv('/kaggle/input/kazakhstan-respa-final-day-2-late-competition/sample_submission.csv')

# ЛОГИКА: Нам нужно найти БОЛЕЕ СТАРЫЙ текст.
# Модель предсказывает дату (ordinal). Чем МЕНЬШЕ число, тем СТАРЕЕ текст.
# Обычно в задаче: Label 1 (или 0) означает "Первый текст старее".
# Допустим, 1 = First is Older.
# Если pred1 < pred2 (дата 1 меньше даты 2), значит 1 старее -> Label = 1.

# Вариант А: Если 1 - это "Первый старше"
sub['label'] = (pred1 < pred2).astype(int)
sub.to_csv("submission_accurate.csv", index=False)

# Вариант Б: Если нужно вероятности (для ROC-AUC), но тут классификация 0/1
# На всякий случай сделай инвертированный файл, если метрика будет 0.00
# sub['label'] = (pred1 > pred2).astype(int)
# sub.to_csv("submission_inverted.csv", index=False)

print("Готово! Файл submission_accurate.csv сохранен.")

2026-02-10 06:06:57.739477: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770703617.931068      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770703617.987628      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770703618.454165      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770703618.454208      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770703618.454211      55 computation_placer.cc:177] computation placer alr

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Step,Training Loss
50,1.027300
100,0.994000
150,0.886900
200,0.861200
250,0.828900
300,0.795500
350,0.807000
400,0.780400
450,0.777800
500,0.764400


Предсказание для test...


Map:   0%|          | 0/24050 [00:00<?, ? examples/s]

Map:   0%|          | 0/24050 [00:00<?, ? examples/s]

Готово! Файл submission_accurate.csv сохранен.
